# 01 — Distributions

Companion to [`../../data_visualization/distributions.md`](../../data_visualization/distributions.md).

---

## Why distribution plots matter

Understanding how a variable is distributed is the **first step** in any data analysis. Distributions tell you:

- **Central tendency** — where is the "typical" value?
- **Spread** — how variable is the data?
- **Shape** — is it symmetric, skewed, multimodal?
- **Outliers** — are there extreme values?
- **Class balance** — for categorical variables, are some classes rare?

This notebook covers the essential distribution plots, when to use each, and how to compare groups.

All data used is drawn from the handbook's [`sample_data`](../sample_data/) directory with industry-relevant context.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', font_scale=1.0)
rng = np.random.default_rng(42)

print('Libraries loaded.')

## 1. Histogram — first look at data shape

Loading real sample data: **Product A testing scores** from `sample_data/distributions/histogram.csv`.
This simulates a common scenario in **manufacturing quality control** where we track test scores across batches.

In [ ]:
hist_data = pd.read_csv('../sample_data/distributions/histogram.csv')
hist_data

In [ ]:
# Reconstruct individual values from binned data for histogram
values = []
for _, row in hist_data.iterrows():
    values.extend([row['bin_start'], row['bin_end']] * row['count'])
values = np.array(values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram with counts
sns.histplot(hist_data, x='bin_start', weights='count', bins=8, ax=axes[0], 
             color='steelblue', edgecolor='white', kde=True)
axes[0].set_xlabel('Score Range Start'); axes[0].set_ylabel('Count')
axes[0].set_title('Product A Test Scores — Histogram + KDE\n(Manufacturing Quality Control)')

# Histogram with density
sns.histplot(values, bins=15, ax=axes[1], kde=True, color='teal', stat='density')
axes[1].set_xlabel('Score'); axes[1].set_ylabel('Density')
axes[1].set_title('Reconstructed Distribution — Density View')

plt.tight_layout(); plt.show()
print(f'Mean: {values.mean():.1f} | Median: {np.median(values):.1f} | Std: {values.std():.1f}')

## 2. Density plot (KDE) — smooth distribution

Using `sample_data/distributions/density_plot.csv` — comparing **Product A vs Product B** performance distributions.
This is a common pattern in **retail/e-commerce** A/B testing of product variants.

In [ ]:
density_data = pd.read_csv('../sample_data/distributions/density_plot.csv')
density_data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overlaid KDE by product
sns.kdeplot(data=density_data, x='value', hue='product', ax=axes[0], 
            fill=True, alpha=0.3, linewidth=2)
axes[0].set_xlabel('Performance Score'); axes[0].set_ylabel('Density')
axes[0].set_title('Product A vs B — Overlaid KDE\n(Retail Product Testing)')
axes[0].legend(['Product A', 'Product B'])

# Side-by-side KDE
for i, prod in enumerate(['Product A', 'Product B']):
    subset = density_data[density_data['product'] == prod]['value']
    sns.kdeplot(subset, ax=axes[1], linewidth=2, label=prod)
    axes[1].fill_between(axes[1].get_xlim(), 0, 0, alpha=0)

axes[1].set_xlabel('Performance Score'); axes[1].set_ylabel('Density')
axes[1].set_title('Product A vs B — Side-by-Side KDE')
axes[1].legend()

plt.tight_layout(); plt.show()

## 3. Box plot — outliers and quartiles

Using `sample_data/distributions/box_plot.csv` — comparing **revenue distributions across product categories**.
This mirrors a **retail analytics** scenario where we compare performance across categories.

In [ ]:
box_data = pd.read_csv('../sample_data/distributions/box_plot.csv')
box_data

In [ ]:
# Create synthetic individual points from summary stats for realistic box plot
all_points = []
all_cats = []
for _, row in box_data.iterrows():
    # Generate points from min to max using the quartile structure
    n_per_cat = 100
    np.random.seed(42 + hash(row['category']))
    points = np.sort(np.random.uniform(row['min'], row['max'], n_per_cat))
    all_points.extend(points)
    all_cats.extend([row['category']] * n_per_cat)

box_df = pd.DataFrame({'category': all_cats, 'value': all_points})

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=box_df, x='category', y='value', palette='Set2', width=0.5)
sns.stripplot(data=box_df, x='category', y='value', color='black', alpha=0.15, size=3)
ax.set_xlabel('Product Category'); ax.set_ylabel('Revenue ($)')
ax.set_title('Revenue Distribution by Category — Box Plot + Strip\n(Retail Analytics)')
plt.tight_layout(); plt.show()

print('Summary statistics:')
for _, row in box_data.iterrows():
    print(f"  {row['category']}: median={row['median']}, IQR={row['q3']-row['q1']}, "
          f"outliers above Q3+1.5*IQR: {row['max'] > row['q3'] + 1.5*(row['q3']-row['q1'])}")

## 4. Violin plot — density + distribution shape

Using `sample_data/distributions/violin_plot.csv` — **income distributions across neighborhoods**.
This is a classic **public sector / urban planning** analysis pattern.

In [ ]:
violin_data = pd.read_csv('../sample_data/distributions/violin_plot.csv')
violin_data.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Violin plot
sns.violinplot(data=violin_data, x='location', y='income', ax=axes[0], 
               palette='Set2', inner='quartile')
axes[0].set_xlabel('Location'); axes[0].set_ylabel('Income ($k)')
axes[0].set_title('Income by Location — Violin Plot\n(Urban Planning / Public Sector)')

# Violin + swarm combo
sns.violinplot(data=violin_data, x='location', y='income', ax=axes[1], 
               palette='Set2', inner=None, alpha=0.4)
sns.swarmplot(data=violin_data, x='location', y='income', ax=axes[1], 
              color='black', alpha=0.3, size=3)
axes[1].set_xlabel('Location'); axes[1].set_ylabel('Income ($k)')
axes[1].set_title('Income by Location — Violin + Swarm\n(Density + individual points)')

plt.tight_layout(); plt.show()

## 5. Stem-and-leaf plot — preserve individual values

Using `sample_data/distributions/stem_leaf.csv` — **student test scores**.
This is useful in **education analytics** where you want to see both shape and exact values.

In [ ]:
stem_data = pd.read_csv('../sample_data/distributions/stem_leaf.csv')
stem_data

In [ ]:
# Build a stem-and-leaf display manually
scores = stem_data['test_score'].sort_values().values

stems = {}
for s in scores:
    stem = s // 10
    leaf = s % 10
    stems.setdefault(stem, []).append(leaf)

print('Stem-and-Leaf Plot — Student Test Scores')
print('=' * 30)
for stem in sorted(stems):
    leaves = ''.join(str(l) for l in sorted(stems[stem]))
    print(f"  {stem} | {leaves}")
print('=' * 30)
print(f'N = {len(scores)} | Range = {scores.min()}–{scores.max()}')

# Also show a histogram overlay
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(scores, bins=8, kde=True, color='steelblue', alpha=0.7)
ax.set_xlabel('Test Score'); ax.set_ylabel('Count')
ax.set_title('Test Score Distribution — Stem-and-Leaf + Histogram\n(Education Analytics)')
plt.tight_layout(); plt.show()

## 6. ECDF — the most honest single view

The Empirical CDF plots the fraction of data points ≤ x. No binning, no smoothing — every point matters.

In [ ]:
# Generate realistic income data (log-normal, like real-world income)
income = rng.lognormal(10.5, 0.7, 1000)
segments = rng.choice(['A', 'B', 'C'], 1000, p=[.5, .3, .2])
df = pd.DataFrame({'income': income.round(2), 'segment': segments})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale ECDF
sns.ecdfplot(df, x='income', ax=axes[0], color='steelblue')
axes[0].axhline(0.5, color='red', linestyle='--', alpha=0.5, label='median (y=0.5)')
axes[0].axhline(0.9, color='orange', linestyle='--', alpha=0.5, label='90th percentile')
axes[0].set_title('ECDF — linear scale\n(Median at y=0.5)')
axes[0].set_xlabel('Income'); axes[0].legend()

# Log scale ECDF — tail behavior visible
sns.ecdfplot(df, x='income', ax=axes[1], color='steelblue')
axes[1].set_xscale('log')
axes[1].set_title('ECDF (log x) — tail behavior revealed')
axes[1].set_xlabel('Income (log scale)')

plt.tight_layout(); plt.show()

## 7. Compare distributions across groups

ECDF, box, and violin plots side by side for group comparison.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# ECDF by group
sns.ecdfplot(df, x='income', hue='segment', ax=axes[0], palette='Set2')
axes[0].set_xscale('log')
axes[0].set_title('ECDF by Segment\n(Steeper = more concentrated)')
axes[0].set_xlabel('Income (log)')

# Box plot by group
sns.boxplot(data=df, x='segment', y='income', ax=axes[1], palette='Set2')
axes[1].set_xscale('log')
axes[1].set_title('Box Plot by Segment\n(Outliers as dots)')
axes[1].set_xlabel('')

# Violin + strip
sns.violinplot(data=df, x='segment', y='income', ax=axes[2], palette='Set2', inner='box')
sns.stripplot(data=df, x='segment', y='income', ax=axes[2], color='black', alpha=0.2, size=3)
axes[2].set_xscale('log')
axes[2].set_title('Violin + Strip\n(Density + points)')
axes[2].set_xlabel('')

plt.tight_layout(); plt.show()

## 8. Q-Q plot — checking distributional assumptions

Q-Q plots compare your data's quantiles against a theoretical distribution. If points fall on a straight line, the data follows that distribution.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Q-Q plot vs normal — income is NOT normal
stats.probplot(df['income'], dist='norm', plot=axes[0])
axes[0].set_title('Q-Q vs Normal\n(Curved = not normal)')

# Q-Q plot vs log-normal — income IS log-normal
lognorm_params = stats.lognorm.fit(df['income'].dropna())
stats.probplot(df['income'], dist='lognorm', sparams=lognorm_params, plot=axes[1])
axes[1].set_title('Q-Q vs Log-Normal\n(Straight = good fit!)')

plt.tight_layout(); plt.show()

## 9. Barchart for categorical distributions

Simple but essential: showing frequency of each category.

In [ ]:
counts = df['segment'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Horizontal bar chart
bars = axes[0].barh(counts.index, counts.values, color=sns.color_palette('Set2', 3))
axes[0].set_title('Segment Counts')
axes[0].set_xlabel('Count')
for bar, v in zip(bars, counts.values):
    axes[0].text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2, str(v), va='center')

# Pie chart
axes[1].pie(counts, labels=counts.index, autopct='%1.0f%%', 
            colors=sns.color_palette('Set2', 3))
axes[1].set_title('Segment Proportions')

plt.tight_layout(); plt.show()

## 10. Swarm plot — every data point visible

Great for small-to-medium datasets where you want to see every observation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log scale swarm
sns.swarmplot(data=df, x='segment', y='income', ax=axes[0], palette='Set2', size=4)
axes[0].set_yscale('log')
axes[0].set_title('Swarm Plot (log y)\n(Every point visible)')
axes[0].set_xlabel('')

# Swarm + violin combo
sns.violinplot(data=df, x='segment', y='income', ax=axes[1], palette='Set2', inner=None, alpha=0.3)
sns.swarmplot(data=df, x='segment', y='income', ax=axes[1], palette='Set2', size=3)
axes[1].set_yscale('log')
axes[1].set_title('Violin + Swarm\n(Density + individual points)')
axes[1].set_xlabel('')

plt.tight_layout(); plt.show()

## Takeaways

| Plot | Best for | Caveat |
|------|----------|--------|
| **Histogram** | Quick first look | Bin width choice affects shape |
| **KDE** | Smooth density estimate | Can smooth away real features |
| **ECDF** | Honest, assumption-free view | Less intuitive for non-technical audiences |
| **Box plot** | Outlier detection + IQR | Hides the underlying shape |
| **Violin** | Comparing group distributions | Needs more data per group |
| **Q-Q plot** | Checking distributional assumptions | Requires knowing the target distribution |
| **Swarm** | Small datasets, no overplotting | Scales poorly beyond ~1000 points |
| **Stem-leaf** | Small datasets with exact values | Becomes unwieldy with large data |

**Golden rule**: For comparing groups, ECDF + violin + strip-plot together is unbeatable. Each reveals something the others hide.